# 🏆 Optimized Full Pipeline - FinanceRAG

## 📋 Overview

Pipeline cuối cùng áp dụng **TẤT CẢ các tham số tối ưu** từ notebooks 1 & 2:

### 🎯 Key Features:

1. **Dataset-Specific Hybrid Alpha** 🔍 (from Notebook 1)
   - ConvFinQA: alpha=0.1 (BM25-heavy)
   - FinanceBench: alpha=0.5 (Balanced)
   - FinDER: alpha=0.4 (BM25-heavy)
   - FinQA: alpha=0.9 (Dense-heavy)
   - FinQABench: alpha=0.1 (BM25-heavy)
   - MultiHeirTT: alpha=0.1 (BM25-heavy)
   - TATQA: alpha=0.5 (Balanced)

2. **Dataset-Specific Reranker** 🤖 (from Notebook 2)
   - ConvFinQA: BGE-reranker-base
   - FinanceBench: BGE-reranker-v2-m3
   - FinDER: BGE-reranker-base
   - FinQA: BGE-reranker-v2-m3
   - FinQABench: BGE-reranker-v2-m3
   - MultiHeirTT: BGE-reranker-large
   - TATQA: BGE-reranker-large

3. **Embedding Model**: E5-small (best from notebook 0)
4. **Chunking**: Pre-chunked optimal corpus (from notebook 4)
5. **E5 Prefixes**: query: / passage:

---

## 1. Setup & Imports

In [1]:
# Core Libraries
import os
import sys
import json
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from typing import List, Dict, Tuple
from pathlib import Path

# Embedding & Retrieval
from sentence_transformers import SentenceTransformer
import faiss

# Reranking
from FlagEmbedding import FlagReranker

# BM25
from rank_bm25 import BM25Okapi

# PyTorch
import torch
import gc

# Utils
import warnings
warnings.filterwarnings('ignore')
import logging
logging.disable(logging.CRITICAL)

# Add parent directory to path
sys.path.insert(0, '..')

# Check GPU
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = 'cuda'
else:
    print("Running on CPU")
    device = 'cpu'

print("✅ Libraries imported!")

CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
✅ Libraries imported!


## 2. Configuration with Optimized Parameters

In [2]:
# ============================================================================
# OPTIMIZED CONFIGURATION (from Notebooks 1 & 2)
# ============================================================================

# Base paths
DATA_DIR = Path('../../data')
OUTPUT_DIR = Path('./output_optimized')
CHUNKED_CORPUS_DIR = Path('../../data/chunked_corpus')
CHUNKING_CONFIG_FILE = Path('../../data/chunked_corpus/best_chunking_config_per_dataset.json')

# Create output directory
OUTPUT_DIR.mkdir(exist_ok=True)

# Datasets
DATASETS = [
    'convfinqa',
    'financebench',
    'finder',
    'finqa',
    'finqabench',
    'multiheirtt',
    'tatqa'
]

# Embedding model
EMBEDDING_MODEL = 'intfloat/e5-small-v2'
E5_QUERY_PREFIX = "query: "
E5_PASSAGE_PREFIX = "passage: "

# OPTIMIZED PARAMETERS PER DATASET (from Notebook 1 & 2 results)
OPTIMIZED_CONFIG = {
    'convfinqa': {
        'hybrid_alpha': 0.1,  # BM25-heavy
        'reranker_model': 'BAAI/bge-reranker-base',
        'ndcg_alpha': 0.4385,  # Expected NDCG from notebook 1
        'ndcg_reranker': 0.3175  # Expected NDCG from notebook 2
    },
    'financebench': {
        'hybrid_alpha': 0.5,  # Balanced
        'reranker_model': 'BAAI/bge-reranker-v2-m3',
        'ndcg_alpha': 0.8670,
        'ndcg_reranker': 0.9321
    },
    'finder': {
        'hybrid_alpha': 0.4,  # BM25-heavy
        'reranker_model': 'BAAI/bge-reranker-base',
        'ndcg_alpha': 0.4575,
        'ndcg_reranker': 0.2818
    },
    'finqa': {
        'hybrid_alpha': 0.9,  # Dense-heavy
        'reranker_model': 'BAAI/bge-reranker-v2-m3',
        'ndcg_alpha': 0.4176,
        'ndcg_reranker': 0.4212
    },
    'finqabench': {
        'hybrid_alpha': 0.1,  # BM25-heavy
        'reranker_model': 'BAAI/bge-reranker-v2-m3',
        'ndcg_alpha': 0.8716,
        'ndcg_reranker': 0.9508
    },
    'multiheirtt': {
        'hybrid_alpha': 0.1,  # BM25-heavy
        'reranker_model': 'BAAI/bge-reranker-large',
        'ndcg_alpha': 0.0928,
        'ndcg_reranker': 0.0966
    },
    'tatqa': {
        'hybrid_alpha': 0.5,  # Balanced
        'reranker_model': 'BAAI/bge-reranker-large',
        'ndcg_alpha': 0.4224,
        'ndcg_reranker': 0.4124
    }
}

# Retrieval settings
TOP_K_RETRIEVAL = 100
TOP_K_RERANK = 50
TOP_K_FINAL = 10

# Batch sizes
EMBED_BATCH_SIZE = 32
RERANK_BATCH_SIZE = 16

# Output file
OUTPUT_FILE = OUTPUT_DIR / 'optimized_submission.csv'

# Print configuration
print("\n" + "="*80)
print("⚙️ OPTIMIZED CONFIGURATION")
print("="*80)
print(f"\n📁 Data directory: {DATA_DIR}")
print(f"📁 Chunked corpus: {CHUNKED_CORPUS_DIR}")
print(f"📁 Output: {OUTPUT_DIR}")
print(f"\n🤖 Embedding model: {EMBEDDING_MODEL}")
print(f"🎯 Top-K: Retrieve={TOP_K_RETRIEVAL}, Rerank={TOP_K_RERANK}, Final={TOP_K_FINAL}")

print(f"\n📊 Dataset-Specific Parameters:")
print(f"{'Dataset':<15} {'Alpha':<8} {'Reranker':<25} {'Type':<15}")
print("-"*80)
for ds, cfg in OPTIMIZED_CONFIG.items():
    alpha = cfg['hybrid_alpha']
    reranker = cfg['reranker_model'].split('/')[-1]
    
    if alpha <= 0.3:
        alpha_type = "BM25-heavy 📤"
    elif alpha >= 0.7:
        alpha_type = "Dense-heavy 🎯"
    else:
        alpha_type = "Balanced ⚖️"
    
    print(f"{ds:<15} {alpha:<8} {reranker:<25} {alpha_type:<15}")

print("="*80)


⚙️ OPTIMIZED CONFIGURATION

📁 Data directory: ..\..\data
📁 Chunked corpus: ..\..\data\chunked_corpus
📁 Output: output_optimized

🤖 Embedding model: intfloat/e5-small-v2
🎯 Top-K: Retrieve=100, Rerank=50, Final=10

📊 Dataset-Specific Parameters:
Dataset         Alpha    Reranker                  Type           
--------------------------------------------------------------------------------
convfinqa       0.1      bge-reranker-base         BM25-heavy 📤   
financebench    0.5      bge-reranker-v2-m3        Balanced ⚖️    
finder          0.4      bge-reranker-base         Balanced ⚖️    
finqa           0.9      bge-reranker-v2-m3        Dense-heavy 🎯  
finqabench      0.1      bge-reranker-v2-m3        BM25-heavy 📤   
multiheirtt     0.1      bge-reranker-large        BM25-heavy 📤   
tatqa           0.5      bge-reranker-large        Balanced ⚖️    


## 3. Helper Functions

In [3]:
# ============================================================
# IMPORT SHARED UTILITIES
# ============================================================

from utils import (
    load_jsonl,
    load_prechunked_corpus,
    normalize_scores,
    aggregate_chunk_scores,
    compute_ndcg,
    QRELS_MAPPING
)

print("✅ Utilities imported from utils.py")

✅ Utilities imported from utils.py


In [4]:
# ============================================================
# E5-SPECIFIC FUNCTIONS
# ============================================================

def add_e5_prefix(text: str, is_query: bool = True) -> str:
    """Add E5 prefix to text"""
    prefix = E5_QUERY_PREFIX if is_query else E5_PASSAGE_PREFIX
    return f"{prefix}{text}"


def hybrid_search(query_emb, query_text, faiss_index, bm25, chunk_texts, top_k, alpha=0.6):
    """
    Hybrid search: Dense (alpha) + BM25 (1-alpha)
    
    Args:
        query_emb: Query embedding vector
        query_text: Raw query text (for BM25)
        faiss_index: FAISS index
        bm25: BM25 index
        chunk_texts: List of chunk texts
        top_k: Number of results to return
        alpha: Weight for dense retrieval (0-1)
    
    Returns:
        scores, indices: Combined scores and chunk indices
    """
    # Dense search
    dense_scores, dense_indices = faiss_index.search(
        query_emb.reshape(1, -1).astype('float32'), 
        min(top_k * 2, faiss_index.ntotal)
    )
    dense_scores = dense_scores[0]
    dense_indices = dense_indices[0]
    
    # BM25 search
    query_tokens = query_text.lower().split()
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_top_indices = np.argsort(bm25_scores)[::-1][:top_k * 2]
    
    # Combine candidates
    all_indices = set(dense_indices.tolist()) | set(bm25_top_indices.tolist())
    
    # Normalize scores
    dense_norm = normalize_scores(dense_scores)
    bm25_norm = normalize_scores(bm25_scores[list(all_indices)])
    
    # Create score mapping
    dense_map = {idx: score for idx, score in zip(dense_indices, dense_norm)}
    bm25_map = {idx: bm25_scores[idx] for idx in all_indices}
    bm25_max = max(bm25_map.values()) if bm25_map else 1
    bm25_map = {k: v / bm25_max if bm25_max > 0 else 0 for k, v in bm25_map.items()}
    
    # Combine with alpha weighting
    final_scores = []
    for idx in all_indices:
        d_score = dense_map.get(idx, 0)
        b_score = bm25_map.get(idx, 0)
        final_scores.append((idx, alpha * d_score + (1 - alpha) * b_score))
    
    # Sort and return top-k
    final_scores.sort(key=lambda x: x[1], reverse=True)
    top_results = final_scores[:top_k]
    
    return [s for _, s in top_results], [i for i, _ in top_results]


print("✅ E5 and hybrid search functions defined")

✅ E5 and hybrid search functions defined


## 4. Load Embedding Model

In [5]:
print("Loading embedding model...")
print(f"Model: {EMBEDDING_MODEL}")
embed_model = SentenceTransformer(EMBEDDING_MODEL, device=device)
print("✅ Embedding model loaded!")

# Warm up
_ = embed_model.encode(["test query"], show_progress_bar=False)
print("✅ Model warmed up!")

Loading embedding model...
Model: intfloat/e5-small-v2
✅ Embedding model loaded!
✅ Model warmed up!


## 5. Main Pipeline Function

In [6]:
def process_dataset_optimized(dataset_name: str, dataset_config: Dict):
    """
    Process dataset with OPTIMIZED parameters:
    1. Dataset-specific hybrid alpha
    2. Dataset-specific reranker model
    3. Pre-chunked semantic corpus
    4. E5 prefixes
    """
    print(f"\n{'='*80}")
    print(f"📊 Processing: {dataset_name.upper()}")
    print(f"{'='*80}")
    print(f"🔧 Config: alpha={dataset_config['hybrid_alpha']}, reranker={dataset_config['reranker_model'].split('/')[-1]}")
    
    # Step 1: Load pre-chunked corpus
    print(f"\n📂 Loading pre-chunked corpus...")
    chunks, chunking_method = load_prechunked_corpus(
        dataset_name,
        CHUNKED_CORPUS_DIR,
        CHUNKING_CONFIG_FILE
    )
    
    if not chunks:
        print(f"   ❌ Failed to load chunks for {dataset_name}")
        return None, {}
    
    print(f"   ✅ Loaded {len(chunks)} chunks using {chunking_method}")
    
    # Build chunk-to-doc mapping
    chunk_to_doc = {}
    for c in chunks:
        chunk_id = c.get('_id', c.get('chunk_id', ''))
        doc_id = c.get('original_id', c.get('doc_id', chunk_id))
        chunk_to_doc[chunk_id] = doc_id
    
    # Extract texts with E5 prefix
    chunk_texts_raw = [c.get('text', '')[:1024] for c in chunks]  # Raw for BM25
    chunk_texts = [add_e5_prefix(t, is_query=False) for t in chunk_texts_raw]  # With prefix for embedding
    chunk_ids = [c.get('_id', c.get('chunk_id', '')) for c in chunks]
    
    # Step 2: Encode chunks
    print(f"\n🔢 Encoding {len(chunk_texts)} chunks...")
    chunk_embeddings = embed_model.encode(
        chunk_texts,
        batch_size=EMBED_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    # Step 3: Build FAISS index
    print(f"\n🔍 Building FAISS index...")
    index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
    index.add(chunk_embeddings.astype('float32'))
    print(f"   ✅ Index: {index.ntotal} vectors")
    
    # Step 4: Build BM25 index
    print(f"\n🔤 Building BM25 index...")
    tokenized = [t.lower().split() for t in chunk_texts_raw]
    bm25 = BM25Okapi(tokenized)
    alpha = dataset_config['hybrid_alpha']
    print(f"   ⚖️ Hybrid ratio: {alpha:.1f} Dense / {1-alpha:.1f} BM25")
    
    # Free memory
    del chunk_embeddings
    if device == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()
    
    # Step 5: Load queries
    queries_path = DATA_DIR / f"{dataset_name}_queries.jsonl" / "queries.jsonl"
    queries = load_jsonl(queries_path)
    print(f"\n🎯 Processing {len(queries)} queries...")
    
    query_texts_raw = [q.get('text', '') for q in queries]
    query_texts = [add_e5_prefix(t, is_query=True) for t in query_texts_raw]  # With E5 prefix
    query_ids = [q.get('_id', '') for q in queries]
    
    query_embeddings = embed_model.encode(
        query_texts,
        batch_size=EMBED_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    # Step 6: Load dataset-specific reranker
    print(f"\n🤖 Loading reranker: {dataset_config['reranker_model'].split('/')[-1]}...")
    reranker = FlagReranker(dataset_config['reranker_model'], use_fp16=(device=='cuda'))
    print(f"   ✅ Reranker loaded!")
    
    # Step 7: Retrieve & Rerank
    print(f"\n🔎 Hybrid Retrieval + Reranking...")
    print(f"   Config: top_k_retrieval={TOP_K_RETRIEVAL}, top_k_rerank={TOP_K_RERANK}, top_k_final={TOP_K_FINAL}")
    
    results = []
    
    for i, query_id in enumerate(tqdm(query_ids, desc="Processing")):
        query_emb = query_embeddings[i]
        query_text = query_texts_raw[i]  # Raw text for BM25 and reranking
        
        # Hybrid retrieval with dataset-specific alpha
        scores, chunk_indices = hybrid_search(
            query_emb, query_text, index, bm25, chunk_texts_raw,
            TOP_K_RETRIEVAL, alpha
        )
        
        # Aggregate chunks to documents
        doc_scores = {}
        for idx, score in zip(chunk_indices, scores):
            if idx < 0 or idx >= len(chunk_ids):
                continue
            doc_id = chunk_to_doc.get(chunk_ids[idx], chunk_ids[idx])
            if doc_id not in doc_scores:
                doc_scores[doc_id] = []
            doc_scores[doc_id].append(float(score))
        
        doc_agg = aggregate_chunk_scores(doc_scores, 'max')
        sorted_docs = sorted(doc_agg.items(), key=lambda x: x[1], reverse=True)[:TOP_K_RERANK]
        
        # Prepare reranking candidates
        candidate_ids = [d[0] for d in sorted_docs]
        candidate_texts = []
        for doc_id in candidate_ids:
            # Get all chunks for this document
            doc_chunks = [chunk_texts_raw[j] for j, cid in enumerate(chunk_ids) if chunk_to_doc.get(cid) == doc_id]
            candidate_texts.append(' '.join(doc_chunks)[:2048])
        
        # Rerank with dataset-specific reranker
        pairs = [[query_text, t] for t in candidate_texts]
        rerank_scores = reranker.compute_score(pairs)
        
        if not isinstance(rerank_scores, list):
            rerank_scores = [rerank_scores]
        
        scored = list(zip(candidate_ids, rerank_scores))
        scored.sort(key=lambda x: x[1], reverse=True)
        
        # Store top-k final results
        for doc_id, score in scored[:TOP_K_FINAL]:
            results.append({
                'query_id': query_id,
                'corpus_id': doc_id,
                'score': float(score)
            })
    
    results_df = pd.DataFrame(results)
    print(f"   ✅ Generated {len(results_df)} results")
    
    # Step 8: Evaluate (if qrels available)
    eval_metrics = {}
    qrels_path = DATA_DIR / QRELS_MAPPING.get(dataset_name, f"{dataset_name}_qrels.tsv")
    
    if qrels_path.exists():
        print(f"\n📊 Evaluating...")
        qrels_df = pd.read_csv(qrels_path, sep='\t')
        
        # Build ground truth as dict with relevance scores
        query_col = 'query-id' if 'query-id' in qrels_df.columns else 'query_id'
        corpus_col = 'corpus-id' if 'corpus-id' in qrels_df.columns else 'corpus_id'
        score_col = 'score' if 'score' in qrels_df.columns else 'relevance'
        
        ground_truth = {}
        for _, row in qrels_df.iterrows():
            qid = str(row[query_col])
            cid = str(row[corpus_col])
            relevance = int(row[score_col]) if score_col in qrels_df.columns else 1
            
            if qid not in ground_truth:
                ground_truth[qid] = {}
            ground_truth[qid][cid] = relevance
        
        # Compute NDCG per query
        ndcg_scores = []
        for qid, group in results_df.groupby('query_id'):
            qid_str = str(qid)
            if qid_str in ground_truth:
                retrieved = group['corpus_id'].astype(str).tolist()[:10]
                relevance_dict = ground_truth[qid_str]  # Already a dict
                ndcg = compute_ndcg(retrieved, relevance_dict, k=10)
                ndcg_scores.append(ndcg)
        
        avg_ndcg = np.mean(ndcg_scores) if ndcg_scores else 0.0
        eval_metrics = {
            'NDCG@10': avg_ndcg,
            'num_queries': len(ndcg_scores)
        }
        print(f"   NDCG@10: {avg_ndcg:.4f} (vs expected {dataset_config['ndcg_reranker']:.4f})")
    
    # Clean up
    del query_embeddings, index, bm25, reranker, chunks, tokenized
    if device == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()
    
    return results_df, eval_metrics


print("✅ Pipeline function defined")


✅ Pipeline function defined


## 6. Run Optimized Pipeline on All Datasets

In [7]:
all_results = []
all_eval = {}
failed = []

print("\n" + "="*80)
print("🚀 STARTING OPTIMIZED FULL PIPELINE")
print("="*80)

print("\n📋 Strategy:")
print("   ✅ Use best hybrid alpha per dataset (from notebook 1)")
print("   ✅ Use best reranker per dataset (from notebook 2)")
print("   ✅ Pre-chunked semantic corpus (from notebook 4)")
print("   ✅ E5-small with query:/passage: prefixes")

for dataset in DATASETS:
    try:
        df_res, metrics = process_dataset_optimized(
            dataset,
            OPTIMIZED_CONFIG[dataset]
        )
        
        if df_res is not None:
            all_results.append(df_res)
            if metrics:
                all_eval[dataset] = metrics
        else:
            failed.append(dataset)
            
    except Exception as e:
        print(f"\n❌ Error processing {dataset}: {e}")
        import traceback
        traceback.print_exc()
        failed.append(dataset)

print(f"\n{'='*80}")
print(f"✅ Pipeline completed: {len(all_results)}/{len(DATASETS)} datasets")
if failed:
    print(f"❌ Failed: {failed}")
print(f"{'='*80}")


🚀 STARTING OPTIMIZED FULL PIPELINE

📋 Strategy:
   ✅ Use best hybrid alpha per dataset (from notebook 1)
   ✅ Use best reranker per dataset (from notebook 2)
   ✅ Pre-chunked semantic corpus (from notebook 4)
   ✅ E5-small with query:/passage: prefixes

📊 Processing: CONVFINQA
🔧 Config: alpha=0.1, reranker=bge-reranker-base

📂 Loading pre-chunked corpus...
  ✅ Loaded 38909 pre-chunked chunks
  📋 Method used: unknown
   ✅ Loaded 38909 chunks using unknown

🔢 Encoding 38909 chunks...


Batches:   0%|          | 0/1216 [00:00<?, ?it/s]


🔍 Building FAISS index...
   ✅ Index: 38909 vectors

🔤 Building BM25 index...
   ⚖️ Hybrid ratio: 0.1 Dense / 0.9 BM25

🎯 Processing 421 queries...


Batches:   0%|          | 0/14 [00:00<?, ?it/s]


🤖 Loading reranker: bge-reranker-base...
   ✅ Reranker loaded!

🔎 Hybrid Retrieval + Reranking...
   Config: top_k_retrieval=100, top_k_rerank=50, top_k_final=10


...


🔍 Building FAISS index...
   ✅ Index: 21196 vectors

🔤 Building BM25 index...
   ⚖️ Hybrid ratio: 0.5 Dense / 0.5 BM25

🎯 Processing 1663 queries...


Batches:   0%|          | 0/52 [00:00<?, ?it/s]


🤖 Loading reranker: bge-reranker-large...
   ✅ Reranker loaded!

🔎 Hybrid Retrieval + Reranking...
   Config: top_k_retrieval=100, top_k_rerank=50, top_k_final=10


Processing:   0%|          | 0/1663 [00:00<?, ?it/s]

   ✅ Generated 16630 results

📊 Evaluating...
   NDCG@10: 0.4859 (vs expected 0.4124)

✅ Pipeline completed: 7/7 datasets


## 7. Evaluation Summary

In [8]:
if all_eval:
    print("\n" + "="*80)
    print("📊 EVALUATION SUMMARY (NDCG@10)")
    print("="*80)
    
    print(f"\n{'Dataset':<15} {'Actual':<10} {'Expected':<10} {'Diff':<10} {'Status'}")
    print("-"*80)
    
    total_ndcg = 0
    total_queries = 0
    
    for ds in DATASETS:
        if ds in all_eval:
            actual = all_eval[ds]['NDCG@10']
            expected = OPTIMIZED_CONFIG[ds]['ndcg_reranker']
            diff = actual - expected
            
            if abs(diff) < 0.02:
                status = "✅ Match"
            elif diff > 0:
                status = "🎉 Better!"
            else:
                status = "⚠️ Lower"
            
            print(f"{ds:<15} {actual:<10.4f} {expected:<10.4f} {diff:+10.4f} {status}")
            
            total_ndcg += actual * all_eval[ds]['num_queries']
            total_queries += all_eval[ds]['num_queries']
    
    print("-"*80)
    
    if total_queries > 0:
        avg_ndcg = total_ndcg / total_queries
        print(f"\n📈 WEIGHTED AVERAGE NDCG@10: {avg_ndcg:.4f}")
        print(f"{'='*80}")
        
        # Performance tier
        if avg_ndcg >= 0.55:
            print(f"\n🏆 EXCELLENT! Top-tier performance with optimized parameters!")
        elif avg_ndcg >= 0.45:
            print(f"\n✅ VERY GOOD! Strong results with dataset-specific tuning!")
        elif avg_ndcg >= 0.35:
            print(f"\n✅ GOOD! Solid improvement from optimization!")
        else:
            print(f"\n⚠️ Results lower than expected - check if corpus/queries match")
else:
    print("\n⚠️ No evaluation metrics available")


📊 EVALUATION SUMMARY (NDCG@10)

Dataset         Actual     Expected   Diff       Status
--------------------------------------------------------------------------------
convfinqa       0.3995     0.3175        +0.0820 🎉 Better!
financebench    0.8720     0.9321        -0.0601 ⚠️ Lower
finder          0.4604     0.2818        +0.1786 🎉 Better!
finqa           0.4525     0.4212        +0.0313 🎉 Better!
finqabench      0.8716     0.9508        -0.0792 ⚠️ Lower
multiheirtt     0.0979     0.0966        +0.0013 ✅ Match
tatqa           0.4859     0.4124        +0.0735 🎉 Better!
--------------------------------------------------------------------------------

📈 WEIGHTED AVERAGE NDCG@10: 0.4084

✅ GOOD! Solid improvement from optimization!


## 8. Generate Submission File

In [9]:
if all_results:
    final_df = pd.concat(all_results, ignore_index=True)
    submission_df = final_df[['query_id', 'corpus_id']]
    
    # Save
    submission_df.to_csv(OUTPUT_FILE, index=False)
    
    print(f"\n✅ Submission saved: {OUTPUT_FILE}")
    print(f"   Total entries: {len(submission_df):,}")
    print(f"   Unique queries: {submission_df['query_id'].nunique():,}")
    
    print(f"\n📋 Sample results:")
    print(submission_df.head(15))
    
    # Validation
    counts = submission_df.groupby('query_id').size()
    print(f"\n🔍 Validation:")
    print(f"   Results per query: {dict(counts.value_counts().sort_index())}")
    
    if (counts == 10).all():
        print(f"   ✅ All queries have exactly 10 results")
    else:
        print(f"   ⚠️ Some queries don't have 10 results")
        print(f"   Queries with != 10 results: {counts[counts != 10].to_dict()}")
else:
    print("\n❌ No results to save")


✅ Submission saved: output_optimized\optimized_submission.csv
   Total entries: 46,710
   Unique queries: 4,671

📋 Sample results:
     query_id  corpus_id
0   qd4982518  dd4b9f7f6
1   qd4982518  dd4c3b9da
2   qd4982518  dd4971510
3   qd4982518  dd4be45d6
4   qd4982518  dd4bd3790
5   qd4982518  dd4bb016e
6   qd4982518  dd4bec0ec
7   qd4982518  dd4b89cbc
8   qd4982518  dd4b87d18
9   qd4982518  dd4c4f7aa
10  qd49795a8  dd4c05bc8
11  qd49795a8  dd4972226
12  qd49795a8  dd4c5937c
13  qd49795a8  dd4c17a26
14  qd49795a8  dd4ba276c

🔍 Validation:
   Results per query: {10: 4671}
   ✅ All queries have exactly 10 results


## 9. Comparison with Expected Results

In [10]:
# Compare results with expectations from notebooks 1 & 2
if all_eval:
    print("\n" + "="*80)
    print("📊 DETAILED COMPARISON: ACTUAL vs EXPECTED")
    print("="*80)
    
    comparison_data = []
    
    for ds in DATASETS:
        if ds in all_eval:
            actual_ndcg = all_eval[ds]['NDCG@10']
            expected_alpha = OPTIMIZED_CONFIG[ds]['ndcg_alpha']  # From notebook 1
            expected_reranker = OPTIMIZED_CONFIG[ds]['ndcg_reranker']  # From notebook 2
            
            comparison_data.append({
                'Dataset': ds,
                'Alpha (Notebook 1)': OPTIMIZED_CONFIG[ds]['hybrid_alpha'],
                'Expected NDCG (NB1)': expected_alpha,
                'Expected NDCG (NB2)': expected_reranker,
                'Actual NDCG': actual_ndcg,
                'Improvement vs NB1': actual_ndcg - expected_alpha,
                'Match with NB2': 'Yes' if abs(actual_ndcg - expected_reranker) < 0.05 else 'No'
            })
    
    comparison_df = pd.DataFrame(comparison_data)
    
    print("\n" + comparison_df.to_string(index=False))
    
    # Save comparison
    comparison_df.to_csv(OUTPUT_DIR / 'results_comparison.csv', index=False)
    print(f"\n✅ Comparison saved to: {OUTPUT_DIR / 'results_comparison.csv'}")


📊 DETAILED COMPARISON: ACTUAL vs EXPECTED

     Dataset  Alpha (Notebook 1)  Expected NDCG (NB1)  Expected NDCG (NB2)  Actual NDCG  Improvement vs NB1 Match with NB2
   convfinqa                 0.1               0.4385               0.3175     0.399480           -0.039020             No
financebench                 0.5               0.8670               0.9321     0.871977            0.004977             No
      finder                 0.4               0.4575               0.2818     0.460441            0.002941             No
       finqa                 0.9               0.4176               0.4212     0.452516            0.034916            Yes
  finqabench                 0.1               0.8716               0.9508     0.871624            0.000024             No
 multiheirtt                 0.1               0.0928               0.0966     0.097868            0.005068            Yes
       tatqa                 0.5               0.4224               0.4124     0.485861        

## 10. Final Summary Report

In [11]:
print("\n" + "="*80)
print("🏆 OPTIMIZED PIPELINE COMPLETED!")
print("="*80)

print("\n✅ Applied Optimizations:")
print("   1. 🔍 Dataset-specific hybrid alpha (from notebook 1)")
print("      - ConvFinQA: 0.1 (BM25-heavy)")
print("      - FinanceBench: 0.5 (Balanced)")
print("      - FinDER: 0.4 (BM25-heavy)")
print("      - FinQA: 0.9 (Dense-heavy)")
print("      - FinQABench: 0.1 (BM25-heavy)")
print("      - MultiHeirTT: 0.1 (BM25-heavy)")
print("      - TATQA: 0.5 (Balanced)")

print("\n   2. 🤖 Dataset-specific reranker (from notebook 2)")
print("      - ConvFinQA, FinDER: BGE-reranker-base")
print("      - FinanceBench, FinQA, FinQABench: BGE-reranker-v2-m3")
print("      - MultiHeirTT, TATQA: BGE-reranker-large")

print("\n   3. 🧠 E5-small embedding model (best from notebook 0)")
print("   4. ✨ Semantic chunking (optimal per dataset from notebook 4)")
print("   5. 📝 E5 prefixes (query: / passage:)")

if all_eval:
    total_ndcg = sum(m['NDCG@10'] * m['num_queries'] for m in all_eval.values())
    total_queries = sum(m['num_queries'] for m in all_eval.values())
    if total_queries > 0:
        avg = total_ndcg / total_queries
        print(f"\n📊 Final Weighted NDCG@10: {avg:.4f}")
        
        # Calculate expected average
        expected_avg = sum(OPTIMIZED_CONFIG[ds]['ndcg_reranker'] for ds in all_eval.keys()) / len(all_eval)
        print(f"📊 Expected NDCG@10: {expected_avg:.4f}")
        
        if avg >= expected_avg - 0.01:
            print(f"✅ Results match or exceed expectations!")
        else:
            print(f"⚠️ Results slightly below expectations (may vary due to sampling)")

print(f"\n💾 Output: {OUTPUT_FILE}")
print(f"\n🚀 Ready for Kaggle submission!")
print("="*80)


🏆 OPTIMIZED PIPELINE COMPLETED!

✅ Applied Optimizations:
   1. 🔍 Dataset-specific hybrid alpha (from notebook 1)
      - ConvFinQA: 0.1 (BM25-heavy)
      - FinanceBench: 0.5 (Balanced)
      - FinDER: 0.4 (BM25-heavy)
      - FinQA: 0.9 (Dense-heavy)
      - FinQABench: 0.1 (BM25-heavy)
      - MultiHeirTT: 0.1 (BM25-heavy)
      - TATQA: 0.5 (Balanced)

   2. 🤖 Dataset-specific reranker (from notebook 2)
      - ConvFinQA, FinDER: BGE-reranker-base
      - FinanceBench, FinQA, FinQABench: BGE-reranker-v2-m3
      - MultiHeirTT, TATQA: BGE-reranker-large

   3. 🧠 E5-small embedding model (best from notebook 0)
   4. ✨ Semantic chunking (optimal per dataset from notebook 4)
   5. 📝 E5 prefixes (query: / passage:)

📊 Final Weighted NDCG@10: 0.4084
📊 Expected NDCG@10: 0.4875
⚠️ Results slightly below expectations (may vary due to sampling)

💾 Output: output_optimized\optimized_submission.csv

🚀 Ready for Kaggle submission!
